# **CorrCLIP Demo**

<div style="text-align: center;">
    <a href='https://arxiv.org/abs/2411.10086' target='_blank'>
        <img src='https://img.shields.io/badge/ArXiv-2411.10086-red?style=flat-square' alt='Paper ID'/>
    </a>
</div>

This is a Google Colab demo to perform segmentation on images with custom category names using GPU.

### Install Packages, Get Code, Download Model.



In [ ]:
!pip install -q ftfy hydra-core
!pip install -q -U iopath

print("⏳ Clone CorrCLIP")
!git clone https://github.com/zdk258/CorrCLIP.git
%cd /content/CorrCLIP

from open_clip import create_model
import torch

print("⏳ Download SAM weight")
!wget https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
print("✅ SAM2")

print('⏳ Download CLIP weight')
clip_type = 'ViT-L-14'
pretrained_type = 'openai'
create_model(clip_type, pretrained=pretrained_type)
print("✅ CLIP")

print('⏳ Download DION weight')
torch.hub.load('facebookresearch/dino:main', 'dino_vitb8', weights_only=False)
print("✅ DION")

### Create CorrCLIP.


In [ ]:
from demo_colab import CorrCLIPInfer
print("⏳ Initializing and loading models")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CorrCLIPInfer(clip_type=pretrained_type, model_type=clip_type, dino_type='dino_vitb8', name_path='./configs/my_name.txt', mask_generator=None, device=device)
model.generate_category_embeddings('./configs/my_name.txt')
print("✅ CorrCLIP")

### Set parameters of SAM2.

In [ ]:
sam_parameters = {
    "points_per_side": 16,
    "pred_iou_thresh": 0.4,
    "stability_score_thresh": 0.4,
    "multimask_output": False

}
model.seg_sam2_params(**sam_parameters)

### Perform segmentation on images with custom category names.



In [ ]:
from demo_colab import run_segmentation
from demo_colab import show_result
example_list = [
    ["images/Golden Retriever,Husky,background.jpg", "golden retriever,husky,background"],
    ["images/pikachu,eevee,background.jpg", "pikachu,eevee,background"],
    ["images/animals.png", "cheetah, zebra, rhinoceros, elephant, buffalo, giraffe, antelope, lion, leopard, background"],
    ["images/fruit.jpg", "background, banana, pineapple, broccoli, potato, tomato, chili pepper, kiwi, avocado, orange, lemon, strawberry, cherry tomato, parsley, lime"]
]

example_id = 1
image_path, class_names_text = example_list[example_id][0], example_list[example_id][1]

original_image, segmented_image, detected_classes = run_segmentation(image_path, class_names_text, model, device)
show_result(original_image, segmented_image, detected_classes)